# The Newton-Raphson method

&nbsp;[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/themintlab/ExecutableEngineering/blob/main/chapters/root_finding/newton-raphson_method.ipynb)

The efficiency and generalizability of the Newton-Raphson method makes it the go-to root finder for nonlinear system solvers.

> NB: many people abbreviate 'The Newton-Raphson method' to simply 'The Newton method', but this causes confusion with Newton's method for optimization which is related, but a different algorithm.

The Newton-Raphson method has quadratic convergence ($k=2$) *near the root* which is a great result! It does so, however at the cost of calculating the Jacobian and solving a linear system.

As we saw in the previous lecture, Newton's method amounts to using the current position and the (true) tangent to estimate the next guess:

$$ f'(x) \Delta x = - f(x)$$

Graphically:

![newton.png](images/newton.png)

Near the root, it converges quadratically, but there are some not-uncommon scenarios where it can fail.

> Extensions which have greater than quadratic convergence using higher order derivative information do exist, however the additional computational cost is typically on par with another Newton-Raphson step, and therefore they are only used in certain niche applications.

The Newton-Raphson method only finds *local* roots, not all of them. Efficient and robust root finding **requires a good initial guess**.

Fortunately, in Engineering, this is commonly the case!

**Example of an initial guess**

If we need to solve for temperature $T(x,y,z,t)$ as a nonlinear, time dependent, partial differential equation, we will be given an initial value for $T(x,y,z, t=0)$.

When solving a nonlinear equation for $T(x,y,z,t=1)$, what do you suppose the initial guess should be?

Answer: The initial guess should be the solution at the preceding time step!

In [ ]:
import numpy as np
import plotly.graph_objects as go

try:
    import executable_engineering as exe
except ImportError:
    %pip install -q executable_engineering
    import executable_engineering as exe

## Example: find the root of $x^3-2x+2$

There is a real root at x~-1.769.

In [ ]:
def f(x):
  return x**3 - 2*x + 2

def df(x):
  return 3*x**2 - 2

def newton_raphson(f, df, x0, max_iter=8, tolerance=1e-6):
  x = x0
  guesses = [x]
  for i in range(max_iter):
    x_new = x - f(x) / df(x)
    guesses.append(x_new)
    if abs(x_new - x) < tolerance:
      return x_new, guesses
    x = x_new
  return None, guesses

x0 = 2
root, guesses = newton_raphson(f, df, x0)

x_vals = np.linspace(-3, 3, 100)
y_vals = f(x_vals)
fig = go.Figure()
fig.add_trace(go.Scatter(x=x_vals, y=y_vals, mode='lines', name="f(x) = x^3 - 2x + 2"))
for i, guess in enumerate(guesses):
  if i == len(guesses) - 1:
    fig.add_trace(go.Scatter(x=[guess], y=[f(guess)], mode='markers', marker=dict(color='red', size=10), showlegend=False))
  else:
    fig.add_trace(go.Scatter(x=[guess], y=[f(guess)], mode='markers', marker=dict(color='blue', size=8), showlegend=False))
  if i > 0:
    fig.add_trace(go.Scatter(x=[guesses[i-1], guess], y=[f(guesses[i-1]), f(guess)], mode='lines', line=dict(dash='dash', color='green'), showlegend=False))

fig.add_hline(y=0, line_dash='dash', line_color='black')
fig.update_layout(title="Newton-Raphson Method", xaxis_title="x", yaxis_title="f(x)", xaxis_range=[-3, 6], yaxis_range=[-10, 40])
fig.show()

if root:
  print(f"Root found: {root:.6f}")
else:
  print("Newton-Raphson method failed to converge.")


## Example: Find the root of $\sqrt{x}$

In [ ]:
def f(x):
  return np.sqrt(x)

def jacobian(x):
  return .5*x**(-1./2)

root, guesses = newton_raphson(f, jacobian, x0 = .1)
print(root, guesses)


## Some common failure situations

What went wrong in the previous example? What else could we try?

![Untitled.png](images/Untitled.png)

Before we talk about mitigation strategies, let's generalize the Newton-Raphson method to N-D.

# The N-D Newton-Raphson method

The Newton-Raphson method thus far has been described for scalar functions or scalar arguments (i.e.: 1-D).

Consider a system of $n$ unkowns $\vec{x}$ and a set of $n$ nonlinear equations that we wish to solve simultaneously:

$$\begin{align}
f_1(\vec{x}) &= 0 \\
f_2(\vec{x}) &= 0 \\
\vdots \\
f_n(\vec{x}) &= 0
\end{align} $$

which may be summarized as a vector function $\vec{f}(\vec{x})=\vec{0}$ also of dimension $n$. Since we have $n$ equations and $n$ unkowns, we can (hopefully) find an exact solution of the simultaneous set of equations, i.e.: a root.

The Newton-Raphson method generalizes quite readily except the derivative must be replaced by the vector-derivative of a vector function (called the *Jacobian*):

$$
J = \frac{\partial \vec{f}}{\partial \vec{x}} =
\begin{bmatrix}
\frac{\partial f_1}{\partial x_1} & \frac{\partial f_1}{\partial x_2} & \dots & \frac{\partial f_1}{\partial x_n} \\
\frac{\partial f_2}{\partial x_1} & \frac{\partial f_2}{\partial x_2} & \dots & \frac{\partial f_2}{\partial x_n} \\
\vdots & \vdots & \ddots & \vdots \\
\frac{\partial f_n}{\partial x_1} & \frac{\partial f_n}{\partial x_2} & \dots & \frac{\partial f_n}{\partial x_n}
\end{bmatrix}
$$

where we can see that $J$ is a square $n \times n$ matrix. The Newton-Raphson method takes the form:

$$
J \Delta \vec{x} = - \vec{f}
$$

which is .... (wait for it!)... a linear system solving for the vector of increments, $\Delta\vec{x}$!

This is an example of computational-thinking: we have broken down a muiltivariable non-linear vector function into a sequence of linear systems!

## Example: solve a system of nonlinear equations

$$ \begin{align}
x^2 + y^2 - z &= 1 \\
x - y^2 + z^2 &= 1 \\
x y z &= 1
\end{align} $$

Rewrite the equations as a system of nonlinear functions:

$$\vec{f}(\vec{x}) = \begin{bmatrix}
f_1(x, y, z) \\
f_2(x, y, z) \\
f_3(x, y, z)
\end{bmatrix} =
\begin{bmatrix}
x^2 + y^2 - z - 1 \\
x - y^2 + z^2 -1 \\
x y z - 1
\end{bmatrix}
=\vec{0}
$$

The Jacobian is:

$$
J = \frac{\partial \vec{f}}{\partial \vec{x}} =
\begin{bmatrix}
2x & 2y & -1 \\
1 & -2y & 2z \\
yz & xz & xy
\end{bmatrix}
$$

Now for a given $\vec{x}$ we can solve for the increment.

In [ ]:
def f(x):
  x, y, z = x
  return np.array([
      x**2 + y**2 - z - 1,
      x - y**2 + z**2 - 1,
      x * y * z - 1
  ])

def jacobian(x):
  x, y, z = x
  return np.array([
      [2 * x, 2 * y, -1],
      [1, -2 * y, 2 * z],
      [y * z, x * z, x * y]
  ])

def newton_raphson_nd(x0, tolerance=1e-6, max_iterations=100):
  x = x0
  for iter in range(max_iterations):
    f_x = f(x)
    print("Iteration, ", iter, " the guess is ", np.round(x,3), " with residual ", np.linalg.norm(f_x) )
    J_x = jacobian(x)

    delta_x = np.linalg.solve(J_x, -f_x)
    x = x + delta_x
    if np.linalg.norm(f_x) < tolerance:
      return x
  return None

x0 = np.array([2, 2, 2])
solution = newton_raphson_nd(x0)

if solution is not None:
  print("Solution found:", solution)
else:
  print("No solution found within the maximum iterations.")


## Exploring initial guesses

Let's try a few different initial guesses to see how the method behaves:

In [ ]:
print("\n--- Guess [3,3,3] ---")
solution1 = newton_raphson_nd(np.array([3, 3 , 3]))
print("Great, but something is funny with the residual... Let's keep going!")

print("\n--- Guess [10,10,10] ---")
solution2 = newton_raphson_nd(np.array([10, 10 , 10]))
print("Still converged but to a different root...")

print("\n--- Guess [-1,-1,-1] ---")
solution3 = newton_raphson_nd(np.array([-1, -1 ,-1]))
print("Another root?")

print("\n--- Guess [-10,0,-10] ---")
solution4 = newton_raphson_nd(np.array([-10, 0,-10]))
print("Yikes! Note what's happening to guesses and the residual... odd behaviour indeed!")